# Predicting Stellar Class — Modeling PLUS (boosters + RealMLP + blend)

One self-contained notebook. It:
1. loads `train`/`test` and (optionally) appends external **SDSS17** (cleaned of `-9999`);
2. trains **LightGBM + XGBoost + CatBoost** on our engineered features;
3. trains a **RealMLP-TD** (architecture reused verbatim from the public `nb02`) on its own
   floor-categorized + target-encoded features — also using the external data;
4. blends all four models with a **simple weighted blender** (hill climbing on OOF), tunes
   **per-class multipliers** for balanced accuracy, and writes `submission.csv`.

All models share one `StratifiedKFold(5, shuffle=True, random_state=42)` split over the real
rows (external rows join training only), so every OOF is honest and aligned in-process.
Heavy: boosters ~0.5 h + RealMLP ~3 h on a Kaggle GPU.

## 0. Config & imports

In [ ]:
import warnings, time, gc, os, sys, math, random, glob, json
warnings.filterwarnings('ignore')
from contextlib import contextmanager
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import TargetEncoder
from sklearn.utils.class_weight import compute_class_weight


@contextmanager
def quiet():
    '''Redirect C-level stdout/stderr (fd 1 & 2) to devnull (hides native GBM GPU chatter).'''
    devnull = os.open(os.devnull, os.O_WRONLY)
    saved = [os.dup(1), os.dup(2)]
    try:
        os.dup2(devnull, 1); os.dup2(devnull, 2)
        yield
    finally:
        os.dup2(saved[0], 1); os.dup2(saved[1], 2)
        os.close(devnull); os.close(saved[0]); os.close(saved[1])

SEED = 42
N_SPLITS = 5
TARGET = 'class'
ID = 'id'
CLASSES = ['GALAXY', 'QSO', 'STAR']
NC = len(CLASSES)
n_classes = NC
class_to_int = {c: i for i, c in enumerate(CLASSES)}
int_to_class = {i: c for c, i in class_to_int.items()}

USE_GPU = True          # boosters on GPU where available
USE_EXTERNAL = True     # append external SDSS17 (add it as a Kaggle input / docs/external/)
MLP_USE_EXTERNAL = False  # RealMLP excludes external rows (their NaN spectral/population
                          # category is absent in test and pollutes the net's embeddings)
MLP_SEEDS = [42, 1337, 2024]  # multi-seed RealMLP: average to denoise the dominant model

ON_KAGGLE = Path('/kaggle/input').exists()
if ON_KAGGLE:
    DATA_DIR = sorted(Path('/kaggle/input').rglob('train.csv'))[0].parent
    OUT_DIR = Path('/kaggle/working')
else:
    DATA_DIR = Path('../../docs/dataset')
    OUT_DIR = Path('../../data_processed')
print('Env:', 'Kaggle' if ON_KAGGLE else 'local', '| DATA_DIR:', DATA_DIR)

## GPU bootstrap (RealMLP / torch)

Installs a Pascal-compatible torch build if the session lands on a P100; no-op on T4 / CPU.
Defines `CUDA_OK`, used by the RealMLP config.

In [ ]:
import subprocess, sys
def _gpu_name():
    try:
        return subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
            text=True).strip()
    except Exception:
        return ''
_NAME = _gpu_name()
print('Detected GPU:', _NAME or '(none / CPU)')
# Pascal P100 = compute capability 6.0, unsupported by the cu128 wheel.
if 'P100' in _NAME:
    print('P100 detected -> installing Pascal-compatible torch (cu121) ...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'torch==2.5.1', '--index-url',
                    'https://download.pytorch.org/whl/cu121'], check=False)
import torch
# verify the GPU can actually launch a kernel; otherwise fall back to CPU
CUDA_OK = False
if torch.cuda.is_available():
    try:
        (torch.zeros(8, device='cuda') + 1).sum().item()
        CUDA_OK = True
        print('CUDA kernel test OK on', torch.cuda.get_device_name(0))
    except Exception as e:
        print('WARNING: CUDA kernel test failed, using CPU:', e)
print('torch', torch.__version__, '| CUDA_OK', CUDA_OK)

In [ ]:
# torch imports AFTER the P100 bootstrap (so a reinstalled build is the one imported)
import torch
import torch.nn as nn
import torch.nn.functional as F
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 1. Load data

In [ ]:
def reduce_mem(df):
    for c in df.select_dtypes('float64').columns:
        df[c] = df[c].astype('float32')
    for c in df.select_dtypes('int64').columns:
        df[c] = pd.to_numeric(df[c], downcast='integer')
    return df

train = reduce_mem(pd.read_csv(DATA_DIR / 'train.csv'))
test = reduce_mem(pd.read_csv(DATA_DIR / 'test.csv'))
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
print('train:', train.shape, '| test:', test.shape)
train.head(3)

### Append external SDSS17 (cleaned)

In [ ]:
train['is_external'] = 0

if USE_EXTERNAL:
    # On Kaggle the dataset is added as an input; locally we read it from docs/external/.
    ext_paths = sorted(Path('/kaggle/input').rglob('star_classification.csv'))
    if not ext_paths:
        local_ext = DATA_DIR.parent / 'external' / 'star_classification.csv'
        if local_ext.exists():
            ext_paths = [local_ext]
    if ext_paths:
        ext = pd.read_csv(ext_paths[0])
        ext[TARGET] = ext['class'].astype(str).str.upper()
        # SDSS17 uses -9999 as a sentinel for missing photometry -> drop those rows,
        # otherwise the bands get poisoned for the boosters (EDA flagged this).
        band_ok = ~(ext[['u', 'g', 'r', 'i', 'z']] < -50).any(axis=1)
        n_bad = int((~band_ok).sum())
        ext = ext[band_ok].copy()
        # The original set lacks spectral_type / galaxy_population -> mark as missing.
        for c in ['spectral_type', 'galaxy_population']:
            if c not in ext.columns:
                ext[c] = np.nan
        common = [c for c in train.columns if c in ext.columns]
        ext = reduce_mem(ext[common].copy())
        ext['is_external'] = 1
        train = pd.concat([train, ext], ignore_index=True)
        print('Appended external rows:', len(ext), '| dropped sentinel rows:', n_bad,
              '| new train:', train.shape)
    else:
        print('External dataset not found — skipping.')

## 2. Booster feature engineering

In [ ]:
num_cols = ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift']
bands = ['u', 'g', 'r', 'i', 'z']
cat_cols = ['spectral_type', 'galaxy_population']

def add_features(df):
    df = df.copy()
    # All pairwise SDSS colors (band_a - band_b, a<b).
    color_cols = []
    for a in range(len(bands)):
        for b in range(a + 1, len(bands)):
            name = f'{bands[a]}_{bands[b]}'
            df[name] = df[bands[a]] - df[bands[b]]
            color_cols.append(name)
    # redshift transforms (skew taming for cleaner splits / non-tree members).
    df['redshift_log1p'] = np.log1p(df['redshift'].clip(lower=-0.999))
    # Anomaly flags motivated by EDA.
    df['is_neg_redshift'] = (df['redshift'] < 0).astype('int8')
    df['is_star_like_z'] = (df['redshift'].abs() < 0.002).astype('int8')
    # Photometry aggregates.
    df['mag_mean'] = df[bands].mean(axis=1)
    df['mag_std'] = df[bands].std(axis=1)
    return df, color_cols

train_fe, color_cols = add_features(train)
test_fe, _ = add_features(test)

extra_num = ['redshift_log1p', 'is_neg_redshift', 'is_star_like_z', 'mag_mean', 'mag_std']
print('color_cols (%d):' % len(color_cols), color_cols)
print('extra_num:', extra_num)

### Categorical encoding

In [ ]:
spectral_order = {'O/B': 0, 'A/F': 1, 'G/K': 2, 'M': 3}
pop_map = {'Blue_Cloud': 0, 'Red_Sequence': 1}

for df in (train_fe, test_fe):
    df['spectral_type_code'] = df['spectral_type'].map(spectral_order).fillna(-1).astype('int16')
    df['galaxy_population_code'] = df['galaxy_population'].map(pop_map).fillna(-1).astype('int16')
    df['z_x_spectral'] = df['redshift'] * (df['spectral_type_code'] + 1)

feature_cols = (num_cols + color_cols + extra_num +
                ['spectral_type_code', 'galaxy_population_code', 'z_x_spectral'])

for df in (train_fe, test_fe):
    for c in cat_cols:
        df[c + '_cat'] = df[c].astype('object').where(df[c].notna(), 'NA').astype(str)
cat_feature_cols = (num_cols + color_cols + extra_num +
                    ['z_x_spectral', 'spectral_type_cat', 'galaxy_population_cat'])
cb_cat_idx = [cat_feature_cols.index('spectral_type_cat'),
              cat_feature_cols.index('galaxy_population_cat')]

train_fe['target'] = train_fe[TARGET].map(class_to_int).astype('int8')
print('LGBM/XGB features (%d)' % len(feature_cols))
print('CatBoost features (%d), cat idx %s' % (len(cat_feature_cols), cb_cat_idx))
print('null check:', train_fe[feature_cols].isnull().sum().sum(),
      test_fe[feature_cols].isnull().sum().sum())

## 3. CV setup (shared folds)

In [ ]:
y = train_fe['target'].values
is_real = (train_fe['is_external'].values == 0)

counts = np.bincount(y[is_real], minlength=len(CLASSES))
class_w = counts.sum() / (len(CLASSES) * np.maximum(counts, 1))
sample_w = class_w[y].astype('float32')
print('class counts:', counts, '| class weights:', class_w.round(3))

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
ext_idx = np.where(~is_real)[0]
real_idx = np.where(is_real)[0]
folds = []
for tr_r, va_r in skf.split(real_idx, y[real_idx]):
    tr = np.concatenate([real_idx[tr_r], ext_idx])
    va = real_idx[va_r]
    folds.append((tr, va))
print('folds:', [(len(t), len(v)) for t, v in folds])

X = train_fe[feature_cols]
Xc = train_fe[cat_feature_cols]
Xt = test_fe[feature_cols]
Xtc = test_fe[cat_feature_cols]
n_test = len(test_fe)
NC = len(CLASSES)

ARTIFACT_DIR = OUT_DIR / 'modeling_plus_artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
RUN_LOG = []
FOLD_LOG = []

def class_recalls(y_true, pred):
    cm = confusion_matrix(y_true, pred, labels=np.arange(NC))
    denom = np.maximum(cm.sum(axis=1), 1)
    return {f'{cls}_recall': cm[i, i] / denom[i] for i, cls in enumerate(CLASSES)}

def score_row(model, proba, seconds=None, notes=''):
    pred = proba.argmax(1)
    row = {
        'model': model,
        'oof_balanced_accuracy': balanced_accuracy_score(y[real_idx], pred),
        'seconds': seconds,
        'features': len(feature_cols),
        'external_rows': int((~is_real).sum()),
        'use_external': bool(USE_EXTERNAL),
        'use_gpu': bool(USE_GPU),
        'notes': notes,
    }
    row.update(class_recalls(y[real_idx], pred))
    return row

def log_model(model, proba, seconds=None, notes=''):
    row = score_row(model, proba[real_idx], seconds=seconds, notes=notes)
    RUN_LOG.append(row)
    display(pd.DataFrame(RUN_LOG).sort_values('oof_balanced_accuracy', ascending=False))
    return row

def save_probs(name, oof, test_pred):
    np.save(ARTIFACT_DIR / f'oof_{name}.npy', oof[real_idx].astype('float32'))
    np.save(ARTIFACT_DIR / f'test_{name}.npy', test_pred.astype('float32'))

def save_run_logs():
    pd.DataFrame(RUN_LOG).to_csv(ARTIFACT_DIR / 'experiment_summary.csv', index=False)
    pd.DataFrame(FOLD_LOG).to_csv(ARTIFACT_DIR / 'fold_metrics.csv', index=False)


## 4. LightGBM

In [ ]:
import lightgbm as lgb
print('lgbm', lgb.__version__)

lgb_params = dict(
    objective='multiclass', num_class=NC, metric='multi_logloss',
    learning_rate=0.03, num_leaves=127, max_depth=-1,
    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
    min_child_samples=60, reg_alpha=1.0, reg_lambda=1.0,
    n_estimators=4000, random_state=SEED, n_jobs=-1, verbose=-1,
)
if USE_GPU:
    lgb_params.update(device='gpu')

def run_lgb():
    oof = np.zeros((len(train_fe), NC), dtype='float32')
    test_pred = np.zeros((n_test, NC), dtype='float32')
    bar = tqdm(folds, desc='LightGBM', unit='fold')
    for f, (tr, va) in enumerate(bar, 1):
        m = lgb.LGBMClassifier(**lgb_params)
        with quiet():
            m.fit(X.iloc[tr], y[tr], sample_weight=sample_w[tr],
                  eval_set=[(X.iloc[va], y[va])], eval_metric='multi_logloss',
                  callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(0)])
        oof[va] = m.predict_proba(X.iloc[va])
        test_pred += m.predict_proba(Xt) / N_SPLITS
        pred = oof[va].argmax(1)
        ba = balanced_accuracy_score(y[va], pred)
        fold_row = {'model': 'lgb', 'fold': f, 'best_iter': m.best_iteration_,
                    'balanced_accuracy': ba, 'train_rows': len(tr), 'valid_rows': len(va)}
        fold_row.update(class_recalls(y[va], pred))
        FOLD_LOG.append(fold_row)
        bar.set_postfix(fold=f, best_iter=m.best_iteration_, BA=f'{ba:.5f}')
        del m; gc.collect()
    return oof, test_pred

t0 = time.time()
oof_lgb, test_lgb = run_lgb()
elapsed = time.time() - t0
save_probs('lgb', oof_lgb, test_lgb)
log_model('lgb', oof_lgb, elapsed, notes='boosters features + external rows')
print('LGBM OOF BA: %.5f  (%.0fs)' % (balanced_accuracy_score(y[real_idx], oof_lgb[real_idx].argmax(1)), elapsed))


## 5. XGBoost

In [ ]:
import xgboost as xgb
print('xgb', xgb.__version__)

xgb_params = dict(
    objective='multi:softprob', num_class=NC, eval_metric='mlogloss',
    learning_rate=0.03, max_depth=8, subsample=0.8, colsample_bytree=0.8,
    min_child_weight=5, reg_alpha=1.0, reg_lambda=2.0,
    n_estimators=4000, random_state=SEED, n_jobs=-1,
)
if USE_GPU:
    xgb_params.update(tree_method='hist', device='cuda')
else:
    xgb_params.update(tree_method='hist')

def run_xgb():
    oof = np.zeros((len(train_fe), NC), dtype='float32')
    test_pred = np.zeros((n_test, NC), dtype='float32')
    bar = tqdm(folds, desc='XGBoost', unit='fold')
    for f, (tr, va) in enumerate(bar, 1):
        m = xgb.XGBClassifier(**xgb_params, early_stopping_rounds=200)
        with quiet():
            m.fit(X.iloc[tr], y[tr], sample_weight=sample_w[tr],
                  eval_set=[(X.iloc[va], y[va])], verbose=False)
        oof[va] = m.predict_proba(X.iloc[va])
        test_pred += m.predict_proba(Xt) / N_SPLITS
        pred = oof[va].argmax(1)
        ba = balanced_accuracy_score(y[va], pred)
        fold_row = {'model': 'xgb', 'fold': f, 'best_iter': m.best_iteration,
                    'balanced_accuracy': ba, 'train_rows': len(tr), 'valid_rows': len(va)}
        fold_row.update(class_recalls(y[va], pred))
        FOLD_LOG.append(fold_row)
        bar.set_postfix(fold=f, best_iter=m.best_iteration, BA=f'{ba:.5f}')
        del m; gc.collect()
    return oof, test_pred

t0 = time.time()
oof_xgb, test_xgb = run_xgb()
elapsed = time.time() - t0
save_probs('xgb', oof_xgb, test_xgb)
log_model('xgb', oof_xgb, elapsed, notes='boosters features + external rows')
print('XGB OOF BA: %.5f  (%.0fs)' % (balanced_accuracy_score(y[real_idx], oof_xgb[real_idx].argmax(1)), elapsed))


## 6. CatBoost (native categoricals)

In [ ]:
from catboost import CatBoostClassifier, Pool
import catboost
print('catboost', catboost.__version__)

cb_params = dict(
    loss_function='MultiClass', eval_metric='TotalF1',
    learning_rate=0.05, depth=8, l2_leaf_reg=5.0,
    iterations=4000, random_seed=SEED, verbose=0,
    auto_class_weights='Balanced',
)
if USE_GPU:
    cb_params.update(task_type='GPU', devices='0')

def run_cb():
    oof = np.zeros((len(train_fe), NC), dtype='float32')
    test_pred = np.zeros((n_test, NC), dtype='float32')
    test_pool = Pool(Xtc, cat_features=cb_cat_idx)
    bar = tqdm(folds, desc='CatBoost', unit='fold')
    for f, (tr, va) in enumerate(bar, 1):
        tr_pool = Pool(Xc.iloc[tr], y[tr], cat_features=cb_cat_idx)
        va_pool = Pool(Xc.iloc[va], y[va], cat_features=cb_cat_idx)
        m = CatBoostClassifier(**cb_params)
        with quiet():
            m.fit(tr_pool, eval_set=va_pool, early_stopping_rounds=200, use_best_model=True)
        oof[va] = m.predict_proba(va_pool)
        test_pred += m.predict_proba(test_pool) / N_SPLITS
        pred = oof[va].argmax(1)
        ba = balanced_accuracy_score(y[va], pred)
        fold_row = {'model': 'cb', 'fold': f, 'best_iter': m.get_best_iteration(),
                    'balanced_accuracy': ba, 'train_rows': len(tr), 'valid_rows': len(va)}
        fold_row.update(class_recalls(y[va], pred))
        FOLD_LOG.append(fold_row)
        bar.set_postfix(fold=f, best_iter=m.get_best_iteration(), BA=f'{ba:.5f}')
        del m, tr_pool, va_pool; gc.collect()
    return oof, test_pred

t0 = time.time()
oof_cb, test_cb = run_cb()
elapsed = time.time() - t0
save_probs('cb', oof_cb, test_cb)
log_model('cb', oof_cb, elapsed, notes='native categoricals + external rows')
print('CatBoost OOF BA: %.5f  (%.0fs)' % (balanced_accuracy_score(y[real_idx], oof_cb[real_idx].argmax(1)), elapsed))


## 7. RealMLP feature engineering

The public RealMLP solution uses its own design: pairwise colors + magnitude aggregates, then
**floor-categorizes every numeric column**, builds two interaction combos, and target-encodes
them per fold. Kept separate from the booster features. Operates on the same `train`/`test`
rows (real + external) in the same order, so it reuses the shared `folds`.

In [ ]:
def build_mlp_features(train_df, test_df):
    BANDS = ['u', 'g', 'r', 'i', 'z']
    def add_feats(df):
        df = df.copy()
        for a, b in [('u','g'),('g','r'),('r','i'),('i','z'),
                     ('u','r'),('u','z'),('g','i'),('g','z'),('r','z'),('u','i')]:
            df[f'{a}_{b}'] = df[a] - df[b]
        M = df[BANDS].values
        df['mag_mean'] = M.mean(1); df['mag_std'] = M.std(1)
        df['mag_min'] = M.min(1); df['mag_max'] = M.max(1)
        z = df['redshift'].values
        df['z_log1p'] = np.log1p(np.clip(z, 0, None))
        df['z_clip'] = np.clip(z, -0.01, 7.0)
        return df

    Xtr = add_feats(train_df); Xte = add_feats(test_df)
    cat_cols = Xtr.select_dtypes(include=['object']).columns.tolist()
    num_cols = Xtr.select_dtypes(exclude=['object']).columns.tolist()
    cmap = {}
    combos = sorted([('alpha_cat_', 'delta_cat_'), ('u_cat_', 'z_cat_')])

    def fe(df, fit):
        for col in cat_cols:
            if fit:
                codes, uniques = df[col].factorize(); cmap[col] = uniques
            else:
                code_map = {c: i for i, c in enumerate(cmap[col])}
                codes = df[col].map(code_map).fillna(-1).astype('int32')
            df[col] = codes; df[col] = df[col].astype('category')
        for col in num_cols:
            cn = f'{col}_cat_'
            if fit:
                codes, uniques = np.floor(df[col]).factorize(); cmap[col] = uniques
            else:
                code_map = {c: i for i, c in enumerate(cmap[col])}
                codes = np.floor(df[col]).map(code_map).fillna(-1).astype('int32')
            df[cn] = codes; df[cn] = df[cn].astype('category')
        combo_names = []
        for cols in combos:
            cn = '_'.join(cols) + '_'; combo_names.append(cn)
            s = df[cols[0]].astype(str)
            for c in cols[1:]:
                s = s + '_' + df[c].astype(str)
            if fit:
                codes, uniques = pd.factorize(s, sort=False); cmap[cn] = uniques
            else:
                code_map = {c: i for i, c in enumerate(cmap[cn])}
                codes = s.map(code_map).fillna(-1).astype('int32')
            df[cn] = codes; df[cn] = df[cn].astype('category')
        new_cat = [c for c in df.columns if c.endswith('_')]
        return df, new_cat, combo_names

    Xtr, new_cat, combo_names = fe(Xtr, True)
    Xte, _, _ = fe(Xte, False)
    all_cat = sorted(cat_cols + new_cat)
    Xtr = Xtr.reindex(sorted(Xtr.columns), axis=1)
    Xte = Xte.reindex(sorted(Xte.columns), axis=1)
    return Xtr, Xte, all_cat, combo_names

# raw frames (drop id/target/marker) -> RealMLP design
mlp_train_raw = train.drop(columns=[c for c in [ID, TARGET, 'is_external'] if c in train.columns])
mlp_test_raw = test.drop(columns=[c for c in [ID] if c in test.columns])
X_mlp, X_mlp_test, mlp_cat_cols, combo_names = build_mlp_features(mlp_train_raw, mlp_test_raw)
print('RealMLP design:', X_mlp.shape, '| cat cols:', len(mlp_cat_cols), '| combos:', combo_names)

## 8. RealMLP model components (verbatim from nb02)

In [ ]:
class NumericalPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, tfms):
        self._tfms = [t for t in tfms if t in ("median_center","robust_scale","smooth_clip","l2_normalize")]
    def fit(self, X, y=None):
        if "median_center" in self._tfms or "robust_scale" in self._tfms:
            self._median = np.median(X, axis=0)
            q = np.quantile(X,0.75,axis=0) - np.quantile(X,0.25,axis=0)
            zi = q == 0.0
            q[zi] = 0.5*(X.max(0)[zi]-X.min(0)[zi])
            self._iqr = 1.0/(q+1e-30); self._iqr[q==0.0] = 0.0
        return self
    def transform(self, X, y=None):
        X = X.copy().astype(np.float32)
        for t in self._tfms:
            if t=="median_center": X -= self._median[None,:]
            elif t=="robust_scale": X *= self._iqr[None,:]
            elif t=="smooth_clip": X = X/np.sqrt(1+(X/3)**2)
            elif t=="l2_normalize":
                n = np.linalg.norm(X,axis=1,keepdims=True); X /= np.where(n==0,1.0,n)
        return X

class CategoricalFeatureLayer(nn.Module):
    def __init__(self, n_ens, cat_dims, embed_dim=8, onehot_thresh=8, device=None):
        super().__init__()
        self.n_ens=n_ens; self.cat_dims=cat_dims; self.onehot_features=[]
        self.embed_layers=nn.ModuleList(); self._embed_feature_indices=[]
        for i,dim in enumerate(cat_dims):
            if dim<=onehot_thresh: self.onehot_features.append(i)
            else:
                self.embed_layers.append(nn.ModuleList([nn.Embedding(dim,embed_dim) for _ in range(n_ens)]))
                self._embed_feature_indices.append(i)
    def forward(self,x):
        b,n_ens,_=x.shape; feats=[]
        if self.onehot_features:
            ox=x[:,:,self.onehot_features]; od=[self.cat_dims[i] for i in self.onehot_features]
            enc=torch.zeros(b,n_ens,sum(od),device=x.device); st=0
            for idx,dim in enumerate(od):
                enc.scatter_(2, ox[:,:,idx:idx+1].long()+st, 1.0); st+=dim
            feats.append(enc)
        for emb_list,fi in zip(self.embed_layers,self._embed_feature_indices):
            fe=[emb_list[mi](x[:,mi,fi:fi+1].long()) for mi in range(self.n_ens)]
            feats.append(torch.cat(fe,dim=1))
        return torch.cat(feats,dim=2)

class ScalingLayer(nn.Module):
    def __init__(self,n_ens,n_features):
        super().__init__(); self.scale=nn.Parameter(torch.ones(n_ens,n_features))
    def forward(self,x): return x*self.scale[None,:,:]

class NTPLinear(nn.Module):
    def __init__(self,n_ens,in_f,out_f,bias=True):
        super().__init__(); self.in_features=in_f
        self.weight=nn.Parameter(torch.randn(n_ens,in_f,out_f))
        self.bias=nn.Parameter(torch.randn(n_ens,out_f)) if bias else None
    def forward(self,x):
        x=torch.einsum("bki,kio->bko",x,self.weight)/math.sqrt(self.in_features)
        if self.bias is not None: x=x+self.bias
        return x

class PBLDEmbedding(nn.Module):
    def __init__(self,n_ens,n_features,hidden_dim=16,out_dim=4,freq_scale=0.1,activation=nn.GELU):
        super().__init__(); self.out_dim=out_dim
        self.w1=nn.Parameter(torch.randn(n_ens,n_features,hidden_dim)*freq_scale)
        self.b1=nn.Parameter(torch.randn(n_ens,n_features,hidden_dim))
        self.w2=nn.Parameter(torch.randn(n_ens,n_features,hidden_dim,out_dim-1)/math.sqrt(hidden_dim))
        self.b2=nn.Parameter(torch.zeros(n_ens,n_features,out_dim-1))
        self.act=activation(); nn.init.uniform_(self.b1,-math.pi,math.pi)
    def forward(self,x):
        periodic=torch.cos(2*math.pi*(x.unsqueeze(-1)*self.w1.unsqueeze(0)+self.b1.unsqueeze(0)))
        transformed=self.act(torch.einsum("bkfh,kfhd->bkfd",periodic,self.w2)+self.b2.unsqueeze(0))
        feat=torch.cat([x.unsqueeze(-1),transformed],dim=-1)
        return feat.flatten(start_dim=2)

class RealMLP(nn.Module):
    def __init__(self,output_dim,cat_dims,n_numerical,cfg):
        super().__init__(); n_ens=cfg["n_ens"]; embed_dim=cfg["embed_dim"]; self.n_ens=n_ens
        self.cate=CategoricalFeatureLayer(n_ens,cat_dims,embed_dim,cfg["onehot_thresh"])
        self.num_embed=PBLDEmbedding(n_ens,n_numerical,cfg["pbld_hidden_dim"],cfg["pbld_out_dim"],cfg["pbld_freq_scale"],cfg["pbld_activation"])
        num_emb=n_numerical*cfg["pbld_out_dim"]
        cat_emb=sum(c if c<=cfg["onehot_thresh"] else embed_dim for c in cat_dims)
        total=num_emb+cat_emb; act=cfg["activation"]; layers=[]
        if cfg["add_front_scale"]: layers.append(ScalingLayer(n_ens,total))
        self._dropout_modules=[]; in_dim=total
        for i,h in enumerate(cfg["hidden_dims"]):
            lin=NTPLinear(n_ens,in_dim,h)
            if i==0: self.first_linear=lin
            d=nn.Dropout(cfg["dropout"]); self._dropout_modules.append(d)
            layers+=[lin,act(),d]; in_dim=h
        self.hidden=nn.Sequential(*layers)
        self.output_layer=NTPLinear(n_ens,in_dim,output_dim)
    def forward(self,x_num,x_cat):
        x_num=x_num.unsqueeze(1).expand(-1,self.n_ens,-1)
        x_cat=x_cat.unsqueeze(1).expand(-1,self.n_ens,-1)
        x=torch.cat([self.num_embed(x_num),self.cate(x_cat)],dim=2)
        return F.softmax(self.output_layer(self.hidden(x)),dim=2)

def apply_schedule(v,p,sched,flat=0.3):
    if sched=="constant": return v
    if sched=="cos": return v*(math.cos(math.pi*p)+1)/2
    if sched=="flat_cos":
        if p<flat: return v
        t=(p-flat)/(1-flat); return v*(math.cos(math.pi*t)+1)/2
    if sched=="flat_anneal":
        if p<flat: return v
        t=(p-flat)/(1-flat); return v*(1-t)
    if sched=="sqrt_cos": return v*math.sqrt((math.cos(math.pi*p)+1)/2)
    if sched=="expm4t": return v*math.exp(-4*p)
    raise ValueError(sched)

def get_parameter_groups(model,p):
    fid=id(model.first_linear.weight); sc,pb,fw,ow,bi=[],[],[],[],[]
    for n,pa in model.named_parameters():
        if "num_embed" in n: pb.append(pa)
        elif "scale" in n: sc.append(pa)
        elif id(pa)==fid: fw.append(pa)
        elif "bias" in n: bi.append(pa)
        else: ow.append(pa)
    LR,WD=p["lr"],p["weight_decay"]
    return [
        {"params":sc,"lr":LR*p["lr_scale_mult"],"weight_decay":WD*p["wd_scale_mult"]},
        {"params":pb,"lr":LR*p["pbld_lr_factor"],"weight_decay":WD},
        {"params":fw,"lr":LR*p["first_layer_lr_factor"],"weight_decay":WD*p["first_layer_wd_factor"]},
        {"params":ow,"lr":LR,"weight_decay":WD},
        {"params":bi,"lr":LR*p["lr_bias_mult"],"weight_decay":WD*p["wd_bias_mult"]},
    ]

def smooth_ce_loss(yt,yp,ls=0.0,cw=None):
    nc=yp.size(1); ys=torch.full_like(yp,ls/nc)
    ys.scatter_(1,yt.unsqueeze(1),1.0-ls+ls/nc)
    loss=-(ys*torch.log(yp.clamp(1e-15,1))).sum(1)
    if cw is not None:
        sw=cw[yt]; return (loss*sw).sum()/sw.sum()
    return loss.mean()

In [ ]:
class RealMLP_TD_Classifier(BaseEstimator):
    def __init__(self, **kw): self.params={**CONFIG, **kw}
    def fit(self, Xtr_df, ytr, Xva_df, yva, cat_col_names=None, ckpt_path="ck.pth", X_test=None):
        p=self.params; dev=torch.device(p["device"] if torch.cuda.is_available() else "cpu")
        cat_col_names=cat_col_names or []
        num_col_names=[c for c in Xtr_df.columns if c not in cat_col_names]
        Xtn=Xtr_df[num_col_names].values.astype(np.float32); Xvn=Xva_df[num_col_names].values.astype(np.float32)
        Xtc=Xtr_df[cat_col_names].values.astype(np.int64); Xvc=Xva_df[cat_col_names].values.astype(np.int64)
        y_tr=np.asarray(ytr); y_v=np.asarray(yva)
        self.preprocessor_=NumericalPreprocessor(p["tfms"]).fit(Xtn)
        Xtn=self.preprocessor_.transform(Xtn); Xvn=self.preprocessor_.transform(Xvn)
        self.cat_col_names_=cat_col_names; self.num_col_names_=num_col_names
        if cat_col_names:
            allc=[Xtc,Xvc]
            if X_test is not None: allc.append(X_test[cat_col_names].values.astype(np.int64))
            cat_dims=(np.concatenate(allc,0).max(0)+1).tolist()
        else: cat_dims=[]
        self.cat_dims_=cat_dims
        if cat_dims:
            cm=np.array(cat_dims)-1; Xtc=np.clip(Xtc,0,cm); Xvc=np.clip(Xvc,0,cm)
        classes=np.unique(y_tr); self.classes_=classes
        cw=torch.as_tensor(compute_class_weight("balanced",classes=classes,y=y_tr),dtype=torch.float32,device=dev)
        self.model_=RealMLP(len(classes),cat_dims,Xtn.shape[1],p).to(dev)
        groups=get_parameter_groups(self.model_,p)
        for g in groups: g["lr_base"]=g["lr"]
        opt=torch.optim.AdamW(groups,betas=(p["mom"],p["sq_mom"]))
        Xtn=torch.as_tensor(Xtn,dtype=torch.float32,device=dev); Xtc=torch.as_tensor(Xtc,dtype=torch.long,device=dev)
        ytt=torch.as_tensor(y_tr,dtype=torch.long,device=dev)
        Xvn=torch.as_tensor(Xvn,dtype=torch.float32,device=dev); Xvc=torch.as_tensor(Xvc,dtype=torch.long,device=dev)
        n_ens=p["n_ens"]; tb=p["train_bs"]; eb=p["eval_bs"]; ep=p["epochs"]
        total=ep*len(y_tr); order=np.arange(len(y_tr)); nc=len(classes)
        best=-np.inf; best_ep=0; self.best_val_probs_=None
        for epoch in range(ep):
            self.model_.train()
            for s in range(0,len(y_tr),tb):
                prog=(epoch*len(y_tr)+s)/total; idx=order[s:s+tb]
                for g in opt.param_groups: g["lr"]=apply_schedule(g["lr_base"],prog,p["lr_sched"],p["flat_ratio"])
                opt.zero_grad(); yp=self.model_(Xtn[idx],Xtc[idx])
                ls=apply_schedule(p["ls_eps"],prog,p["ls_eps_sched"],p["flat_ratio"])
                dr=apply_schedule(p["dropout"],prog,p["p_drop_sched"],p["flat_ratio"])
                for dm in self.model_._dropout_modules: dm.p=dr
                loss=smooth_ce_loss(ytt[idx].repeat_interleave(n_ens),yp.reshape(-1,nc),ls=ls,cw=cw)
                loss.backward(); torch.nn.utils.clip_grad_norm_(self.model_.parameters(),p["grad_clip"]); opt.step()
            np.random.shuffle(order)
            self.model_.eval()
            with torch.no_grad():
                vp=np.concatenate([self.model_(Xvn[s:s+eb],Xvc[s:s+eb]).mean(1).cpu().numpy() for s in range(0,len(y_v),eb)],0)
            sc=balanced_accuracy_score(y_v,vp.argmax(1))
            if sc>best:
                best=sc; best_ep=epoch+1; self.best_val_probs_=vp.copy(); torch.save(self.model_.state_dict(),ckpt_path)
            if p["verbosity"]>=2: print(f"   epoch {epoch+1}/{ep}  ba={sc:.5f}  best={best:.5f}")
        self.model_.load_state_dict(torch.load(ckpt_path)); self.best_score_=best; self._dev=dev
        print(f"   -> best ba {best:.5f} (epoch {best_ep})"); return self
    def predict_proba(self, X):
        eb=self.params["eval_bs"]
        Xn=self.preprocessor_.transform(X[self.num_col_names_].values.astype(np.float32))
        Xc=np.clip(X[self.cat_col_names_].values.astype(np.int64),0,np.array(self.cat_dims_)-1)
        Xn=torch.as_tensor(Xn,dtype=torch.float32,device=self._dev); Xc=torch.as_tensor(Xc,dtype=torch.long,device=self._dev)
        self.model_.eval()
        with torch.no_grad():
            return np.concatenate([self.model_(Xn[s:s+eb],Xc[s:s+eb]).mean(1).cpu().numpy() for s in range(0,len(Xn),eb)],0)

### RealMLP configuration (verbatim from nb02)

In [ ]:
CONFIG = {
    "n_ens": 8, "embed_dim": 7, "onehot_thresh": 10,
    "hidden_dims": [512, 512, 512], "dropout": 0.05, "p_drop_sched": "expm4t",
    "activation": nn.SiLU, "add_front_scale": True,
    "pbld_hidden_dim": 20, "pbld_out_dim": 5, "pbld_freq_scale": 5.0,
    "pbld_activation": nn.PReLU, "pbld_lr_factor": 0.093,
    "lr": 0.01, "mom": 0.9, "sq_mom": 0.98, "lr_sched": "flat_cos", "flat_ratio": 0.3,
    "first_layer_lr_factor": 1.0, "first_layer_wd_factor": 0.1,
    "lr_scale_mult": 10.0, "lr_bias_mult": 0.1, "weight_decay": 0.013,
    "wd_scale_mult": 0.1, "wd_bias_mult": 0.5, "grad_clip": 1.0,
    "ls_eps": 0.04, "ls_eps_sched": "cos",
    "tfms": ["median_center", "robust_scale"],
    "epochs": 8, "train_bs": 256, "eval_bs": 10240, "verbosity": 2,
    "device": "cuda" if CUDA_OK else "cpu", "random_state": 42,
}
FOLDS, SEED = 5, 42
n_classes = 3

## 9. RealMLP — multi-seed, 5-fold (shared folds)

RealMLP is the strongest single model and ~50% of the blend, so we **average several seeds** of
the full 5-fold run (`MLP_SEEDS`) to denoise it. Folds are fixed (shared with the boosters), only
the model seed varies, so `oof_mlp` stays aligned. By default external rows are dropped from
RealMLP training (`MLP_USE_EXTERNAL = False`); validation is always real rows only. Each seed's
OOF BA is printed, followed by the averaged BA — the gap shows the denoising gain.

In [ ]:
mlp_seed_log = []
oof_mlp = np.zeros((len(train_fe), NC), dtype='float32')
test_mlp = np.zeros((n_test, NC), dtype='float32')
is_ext_row = (train_fe['is_external'].values == 1)

t0 = time.time()
for si, seed in enumerate(MLP_SEEDS, 1):
    np.random.seed(seed); random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    cfg = {**CONFIG, 'random_state': seed}
    oof_s = np.zeros((len(train_fe), NC), dtype='float32')
    test_s = np.zeros((n_test, NC), dtype='float32')
    for f, (tr, va) in enumerate(folds, 1):
        # external rows polluted RealMLP's embeddings -> by default train the net on real rows only.
        tr_mlp = tr if MLP_USE_EXTERNAL else tr[~is_ext_row[tr]]
        print(f"\n##### seed {seed} ({si}/{len(MLP_SEEDS)})  fold {f}/{N_SPLITS}  (train rows {len(tr_mlp)}) #####")
        X_tr, X_val, X_tst = X_mlp.iloc[tr_mlp].copy(), X_mlp.iloc[va].copy(), X_mlp_test.copy()

        enc = TargetEncoder(target_type='multiclass', cv=N_SPLITS, smooth='auto',
                            shuffle=True, random_state=SEED)
        tr_e = enc.fit_transform(X_tr[combo_names], y[tr_mlp])
        va_e = enc.transform(X_val[combo_names]); te_e = enc.transform(X_tst[combo_names])
        te_names = [f"_te_{col}_{c}" for col in combo_names for c in range(NC)]
        X_tr[te_names] = tr_e; X_val[te_names] = va_e; X_tst[te_names] = te_e

        model = RealMLP_TD_Classifier(**cfg)
        model.fit(X_tr, y[tr_mlp], X_val, y[va], cat_col_names=mlp_cat_cols,
                  ckpt_path=f"mlp_s{seed}_f{f}.pth", X_test=X_tst)
        oof_s[va] = model.best_val_probs_
        test_s += model.predict_proba(X_tst) / N_SPLITS
        del model; torch.cuda.empty_cache(); gc.collect()

    seed_ba = balanced_accuracy_score(y[real_idx], oof_s[real_idx].argmax(1))
    mlp_seed_log.append({'seed': seed, 'oof_balanced_accuracy': seed_ba})
    print(f"  >> seed {seed} RealMLP OOF BA: {seed_ba:.5f}")
    oof_mlp += oof_s / len(MLP_SEEDS)
    test_mlp += test_s / len(MLP_SEEDS)

elapsed = time.time() - t0
save_probs('mlp', oof_mlp, test_mlp)
pd.DataFrame(mlp_seed_log).to_csv(ARTIFACT_DIR / 'mlp_seed_metrics.csv', index=False)
log_model('mlp', oof_mlp, elapsed, notes=f'RealMLP {len(MLP_SEEDS)}-seed avg; external={MLP_USE_EXTERNAL}')
print('\nRealMLP (%d-seed avg) OOF BA: %.5f  (%.0fs)' %
      (len(MLP_SEEDS), balanced_accuracy_score(y[real_idx], oof_mlp[real_idx].argmax(1)), elapsed))

## 10. Simple blender + per-class tuning

A plain **weighted blend** of the four models' probabilities, with weights found by hill
climbing on the OOF balanced accuracy (start uniform, step the simplex). Then per-class
multipliers before `argmax`. We also print the boosters-only blend for reference.

In [ ]:
yv = y[real_idx]
oofs  = {'lgb': oof_lgb[real_idx], 'xgb': oof_xgb[real_idx],
         'cb': oof_cb[real_idx],  'mlp': oof_mlp[real_idx]}
tests = {'lgb': test_lgb, 'xgb': test_xgb, 'cb': test_cb, 'mlp': test_mlp}
keys = ['lgb', 'xgb', 'cb', 'mlp']
for k in keys:
    print(f'{k:4s} OOF BA: {balanced_accuracy_score(yv, oofs[k].argmax(1)):.5f}')

def blend(ws, source):
    return sum(w * source[k] for w, k in zip(ws, keys))
def ba_of(ws):
    return balanced_accuracy_score(yv, blend(ws, oofs).argmax(1))

# hill-climbing weight search on the simplex (simple, robust)
def hill_climb(active):
    w = np.array([1.0 if k in active else 0.0 for k in keys]); w /= w.sum()
    best = ba_of(w); step = 0.05
    for _ in range(500):
        improved = False
        for i, k in enumerate(keys):
            if k not in active:
                continue
            for d in (step, -step):
                cand = w.copy(); cand[i] += d
                if (cand < -1e-9).any():
                    continue
                cand = cand / cand.sum()
                s = ba_of(cand)
                if s > best + 1e-6:
                    best, w, improved = s, cand, True
        if not improved:
            break
    return w, best

w_gbm, ba_gbm = hill_climb(['lgb', 'xgb', 'cb'])
RUN_LOG.append(score_row('blend_gbm_argmax', blend(w_gbm, oofs), notes='hill-climb blend: lgb+xgb+cb'))
print('boosters-only blend BA: %.5f  w=%s' % (ba_gbm, dict(zip(keys, w_gbm.round(3)))))
w_all, ba_all = hill_climb(keys)
RUN_LOG.append(score_row('blend_all_argmax', blend(w_all, oofs), notes='hill-climb blend: boosters+RealMLP'))
print('boosters + RealMLP  BA: %.5f  w=%s' % (ba_all, dict(zip(keys, w_all.round(3)))))
print('RealMLP contribution: %+.5f' % (ba_all - ba_gbm))

best_w = w_all if ba_all >= ba_gbm else w_gbm
blend_oof = blend(best_w, oofs)
blend_test = blend(best_w, tests)
blend_name = 'blend_all' if ba_all >= ba_gbm else 'blend_gbm'
blend_meta = {
    'keys': keys,
    'weights': {k: float(w) for k, w in zip(keys, best_w)},
    'boosters_only_weights': {k: float(w) for k, w in zip(keys, w_gbm)},
    'all_model_weights': {k: float(w) for k, w in zip(keys, w_all)},
    'boosters_only_oof_ba': float(ba_gbm),
    'all_model_oof_ba': float(ba_all),
    'selected_blend': blend_name,
}
with open(ARTIFACT_DIR / 'blend_meta.json', 'w') as f:
    json.dump(blend_meta, f, indent=2)
oof_blend_full = np.zeros((len(train_fe), NC), dtype='float32')
oof_blend_full[real_idx] = blend_oof.astype('float32')
save_probs('blend_argmax', oof_blend_full, blend_test)
display(pd.DataFrame(RUN_LOG).sort_values('oof_balanced_accuracy', ascending=False))

In [ ]:
def ba_w(proba, mult):
    return balanced_accuracy_score(yv, (proba * mult).argmax(1))

mult = np.ones(NC); base = ba_w(blend_oof, mult)
for _ in range(40):
    improved = False
    for c in range(NC):
        for cand in np.linspace(0.5, 2.0, 31):
            wt = mult.copy(); wt[c] = cand
            s = ba_w(blend_oof, wt)
            if s > base + 1e-6:
                base, mult, improved = s, wt, True
    if not improved:
        break
mult = mult / mult.mean()
print('class multipliers:', dict(zip(CLASSES, mult.round(3))))
print('blend BA argmax  : %.5f' % ba_w(blend_oof, np.ones(NC)))
print('blend BA weighted: %.5f' % base)
print()

final_oof = (blend_oof * mult).argmax(1)
RUN_LOG.append({**score_row('final_blend_weighted', blend_oof * mult,
                            notes='selected blend + per-class multipliers'),
                'class_multipliers': json.dumps({k: float(v) for k, v in zip(CLASSES, mult)})})
print(classification_report(yv, final_oof, target_names=CLASSES, digits=4))
print('confusion matrix (rows=true):')
cm = confusion_matrix(yv, final_oof, labels=np.arange(NC))
cm_df = pd.DataFrame(cm, index=CLASSES, columns=CLASSES)
display(cm_df)

cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(pd.DataFrame(cm_norm, index=CLASSES, columns=CLASSES), annot=True, fmt='.3f', cmap='Blues')
plt.title('Final blend OOF confusion matrix (row-normalized)')
plt.xlabel('predicted'); plt.ylabel('true')
plt.tight_layout()
plt.show()

valid_diag = train_fe.iloc[real_idx][['redshift', 'spectral_type', 'galaxy_population']].copy()
valid_diag['true'] = [int_to_class[i] for i in yv]
valid_diag['pred'] = [int_to_class[i] for i in final_oof]
valid_diag['correct'] = valid_diag['true'].eq(valid_diag['pred'])
valid_diag['error_pair'] = valid_diag['true'] + '->' + valid_diag['pred']
valid_diag['redshift_bin'] = pd.cut(valid_diag['redshift'],
                                    [-np.inf, 0, .01, .05, .1, .25, .5, np.inf])

err = valid_diag[~valid_diag['correct']]
star_gal_err = err[err['error_pair'].isin(['STAR->GALAXY', 'GALAXY->STAR'])]
print('Errors:', len(err), '| STAR/GALAXY boundary errors:', len(star_gal_err))
display(star_gal_err.groupby(['error_pair', 'redshift_bin'], observed=True).size()
        .unstack(fill_value=0))
display(star_gal_err.groupby(['error_pair', 'spectral_type', 'galaxy_population'], observed=True)
        .size().sort_values(ascending=False).head(20).rename('n').to_frame())

fig, ax = plt.subplots(figsize=(9, 4))
star_gal_err.groupby(['redshift_bin', 'error_pair'], observed=True).size().unstack(fill_value=0).plot(kind='bar', ax=ax)
ax.set_title('STAR/GALAXY OOF errors by redshift bin')
ax.set_xlabel('redshift bin'); ax.set_ylabel('OOF error count')
plt.tight_layout()
plt.show()

np.save(ARTIFACT_DIR / 'oof_blend.npy', (blend_oof * mult).astype('float32'))
pd.DataFrame(RUN_LOG).to_csv(ARTIFACT_DIR / 'experiment_summary.csv', index=False)
pd.DataFrame(FOLD_LOG).to_csv(ARTIFACT_DIR / 'fold_metrics.csv', index=False)
valid_diag.to_csv(ARTIFACT_DIR / 'final_oof_diagnostics.csv', index=False)
with open(ARTIFACT_DIR / 'class_multipliers.json', 'w') as f:
    json.dump({k: float(v) for k, v in zip(CLASSES, mult)}, f, indent=2)
print('Saved diagnostics to', ARTIFACT_DIR.resolve())


## 11. Submission

In [ ]:
test_pred = (blend_test * mult).argmax(1)
sub = pd.DataFrame({ID: test_fe[ID].values, TARGET: [int_to_class[i] for i in test_pred]})
assert list(sub.columns) == [ID, TARGET]
assert len(sub) == len(sample_sub)
sub.to_csv(OUT_DIR / 'submission.csv', index=False)

np.save(ARTIFACT_DIR / 'test_blend.npy', (blend_test * mult).astype('float32'))
sub.to_csv(ARTIFACT_DIR / 'submission.csv', index=False)
save_run_logs()
print('Saved', (OUT_DIR / 'submission.csv').resolve(), sub.shape)
print('Artifacts:', ARTIFACT_DIR.resolve())
print(sub[TARGET].value_counts(normalize=True).round(4))
sub.head()
